# Train c3po directly from DANDI

This notebook installs **c3po-neuro** and calls **`c3po.utils.dandi.train_from_dandi` directly in the notebook kernel**. It runs on your own Colab account.

Run the installation cell once, restart the session if requested, then continue from step 2. The package performs data extraction, training, embedding, PCA, and output saving. The current function uses the whole asset, a maximum of 1,000 training epochs with adaptive stopping, and 1-ms context interpolation. These are package defaults; the function does not expose demo mode, epoch limits, batch sizes, seed, or interpolation settings.

The package saves output at the end of training and analysis. Keep this session active until completion, then copy results to Drive or download them.


In [ ]:
# @title 1. Install the package in the notebook runtime
import sys

%pip install "c3po-neuro[gpu]==0.1.0"  "jax==0.11.2" "flax==0.12.10" "dandi"

print("Installation complete. Restart the session, then continue from step 2.")


### Restart after installation
Select **Runtime → Restart session** after installing or replacing packages, especially if Colab requests a restart. Installed packages remain available in that runtime. Continue from **step 2**; do not repeat installation unless you connect to a new runtime.


In [ ]:
# @title 2. Choose a DANDI asset
# Keep leading zeros in the six-digit Dandiset ID.
dandiset_id = "001209"  # @param {type:"string"}
dandi_path = "sub-MonkeyX-held-in-calib/sub-MonkeyX-held-in-calib_ses-20121206_behavior+ecephys.nwb"  # @param {type:"string"}
dandi_instance = "dandi"  # @param ["dandi", "dandi-staging"]
dimensions = 16  # @param {type:"integer"}
save_to_drive = True  # @param {type:"boolean"}

import re

if not re.fullmatch(r"\d{6}", dandiset_id):
    raise ValueError("Use the six-digit Dandiset ID, including leading zeros.")
if not dandi_path.endswith(".nwb"):
    raise ValueError("Enter the path to an NWB asset inside the Dandiset.")
if dimensions < 3:
    raise ValueError("Use at least three dimensions for the default PCA preview.")


In [ ]:
# @title 3. Verify Python, package imports, and GPU
import sys
import importlib.metadata
import os

# Configure JAX before the training module imports it.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
import jax
from c3po.utils.dandi import train_from_dandi

print("Python:", sys.version.split()[0])
print("c3po-neuro:", importlib.metadata.version("c3po-neuro"))
print("JAX:", jax.__version__)
print("Devices:", jax.devices())
if not any(device.platform == "gpu" for device in jax.devices()):
    raise RuntimeError("Select a GPU runtime and confirm JAX detects it before training.")


In [ ]:
# @title 4. Create a run folder and optionally connect Drive
from datetime import datetime, timezone
from pathlib import Path
import json

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = Path("/content/c3po_models") / f"{dandiset_id}_{run_id}"
output_dir.mkdir(parents=True)
drive_output_dir = None

if save_to_drive:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_output_dir = Path("/content/drive/MyDrive/c3po_models") / output_dir.name

run_config = {
    "dandiset_id": dandiset_id,
    "dandi_path": dandi_path,
    "dandi_instance": dandi_instance,
    "dimensions": dimensions,
    "output_dir": str(output_dir),
    "c3po_neuro_version": importlib.metadata.version("c3po-neuro"),
    "python_version": sys.version,
    "jax_version": jax.__version__,
    "jax_devices": [str(device) for device in jax.devices()],
}
(output_dir / "run_config.json").write_text(
    json.dumps(run_config, indent=2), encoding="utf-8"
)
print("Local output:", output_dir)
print("Drive output:", drive_output_dir or "disabled")


### Train
The asset must contain sorted-spike NWB `Units` tables. The package currently uses the default Dandiset version (`draft`). Its settings match the package's DANDI training CLI. Logs, progress bars, and figures appear in this notebook.

If training or saving fails, inspect the error before retrying. Rerun step 4 to create a fresh output folder for a new training run. The package does not overwrite an existing saved model.


In [ ]:
# @title 5. Train directly in this notebook
train_from_dandi(
    dandiset_id=dandiset_id,
    dandi_path=dandi_path,
    dandi_instance=dandi_instance,
    dimensions=dimensions,
    output_dir=output_dir,
)

print(f"Training and analysis completed. Results: {output_dir}")


In [ ]:
# @title 6. Review saved results
from IPython.display import Image, display
import yaml

metrics_path = output_dir / "metrics.yaml"
if metrics_path.exists():
    metrics = yaml.safe_load(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, indent=2))
else:
    raise FileNotFoundError("metrics.yaml is missing; confirm step 5 completed.")

display(Image(filename=str(output_dir / "context_pca_plot.png")))
print("Saved files:")
for path in sorted(output_dir.iterdir()):
    print(f"  {path.name}: {path.stat().st_size / 1e6:.2f} MB")


In [ ]:
# @title 7. Save completed results to Drive
import shutil

if drive_output_dir is not None:
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output_dir, drive_output_dir, dirs_exist_ok=True)
    print(f"Results saved to {drive_output_dir}")
else:
    print("Drive saving is disabled. Use the download cell below.")


In [ ]:
# @title 8. Download the results (optional)
from google.colab import files
import shutil

archive_path = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
files.download(archive_path)


## Notes

- Installation uses `%pip` in the active notebook environment. All c3po computation runs through the direct Python function call in step 5.
- The function saves `params.hdf5`, `args.json`, `embedding.npz`, `context_pca_plot.png`, and `metrics.yaml` in the run folder. The notebook adds `run_config.json`.
- The direct function returns `None`; results are read from its output folder.
- Demo mode, separate training/analysis stages, and the former worker checkpoint mechanism have been removed because these are not exposed by the package's function signature.
- GPU or system-memory errors must be addressed by selecting suitable hardware or changing the package's implementation; this function does not accept batch-size or event-limit overrides.

Validation: the direct call's keywords match the current `c3po.utils.dandi.train_from_dandi` signature. Notebook syntax, parameter validation, and direct-call wiring are checked. Previous real CPU tests covered the package helpers and training/analysis components; this simplified notebook has not been run on a hosted Colab GPU.

Sources: [c3po repository](https://github.com/samuelbray32/c3po), [PyPI package](https://pypi.org/project/c3po-neuro/), [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html).
